# NexPlay — riesgo de arrepentimiento temprano al comprar un videojuego

Diplomado en Ciencia de Datos, FES Acatlán (UNAM) — Módulo V.

Este notebook corre de principio a fin en un Colab limpio: descarga un extracto de datos publicado como asset de un GitHub Release (con verificación SHA-256) y clona el código de entrenamiento del repo a un tag fijo. No usa Google Drive, no pide credenciales y no ejecuta la ingesta de Steam — todo eso ya ocurrió para producir el extracto.

**Narrativa:** Problema → Datos → EDA → Calidad de datos → Ingeniería de variables → Modelo → Experimento de privacidad → Conclusiones.

## 1. Problema

Cuando alguien compra un videojuego en Steam, tiene una ventana de 120 minutos de juego para pedir reembolso. NexPlay busca estimar, **antes de la compra**, el riesgo de que un jugador se arrepienta tempranamente — para eso, antes de que exista una compra real, solo puede usar dos tipos de información: lo que ya se sabe del juego (precio, descuento, recepción de la crítica) y lo que el jugador declara de sí mismo en un formulario de alta.

No observamos arrepentimiento directamente — Steam no pregunta "¿te arrepentiste?". Usamos una señal *proxy*: reseñas donde el autor jugó poco y calificó negativo.

$$Y = 1 \iff \texttt{playtime\_at\_review} < 120 \text{ min} \ \wedge\ \texttt{voted\_up} = 0$$

El umbral de 120 minutos no es arbitrario: es exactamente la ventana de reembolso de Steam. A esta señal la llamamos **arrepentimiento temprano**, nunca "abandono" ni "insatisfacción" — son cosas distintas que esta variable no puede distinguir.

## 2. Datos

### 2.1 Origen

119k+ reseñas ingeridas desde la API pública `appreviews` de Steam sobre un catálogo curado de juegos (`appids.txt` en el repo), pensado en capas:

- **Capa A** — contraste de dificultad/experiencia: juegos que la comunidad adora pero que son duros para alguien nuevo (Dark Souls, Kenshi, Dwarf Fortress) contra puertas de entrada (Stardew Valley, Hades, Portal). Sin este contraste la variable objetivo podría no tener varianza.
- **Capa B** — intersección con el corpus de Metacritic, para comparar motivos entre plataformas (fuera del alcance de este notebook).
- **Capa C** — brecha entre expectativa y recepción: lanzamientos AAA con recibimiento muy disparejo.

### 2.2 Extracto reproducible

`nexplay.db` (SQLite) no se publica: tiene texto de reseñas y vive en `datos/`, fuera de git. En su lugar, `extracto_datos.py` (en el repo) genera un extracto mínimo en Parquet — sin texto, sin `steamid`, sin nada que no haga falta para esta narrativa — y lo publicamos como *asset* de un GitHub Release con **tag fijo** (nunca `latest`, para que esta celda siga funcionando igual dentro de un año). Este notebook descarga ese asset y valida su SHA-256 antes de tocarlo.

El extracto trae: `appid`, `nombre` (para nombrar juegos concretos en el EDA), `playtime_at_review`, `voted_up`, `timestamp_created` — reconstruyen el target y agrupan el GroupKFold — y `num_games_owned`, `es_gratis`, `precio_final`, `descuento`, `metacritic`: las columnas crudas detrás de las seis variables del modelo de producción, más `num_games_owned` cruda (no es feature del modelo, pero sin ella no se puede reproducir la bandera de privacidad de perfil ni el experimento de la sección 7).

In [ ]:
# Dependencias fijadas (misma version que valido este notebook antes de publicarlo).
# Si Colab pide reiniciar el entorno de ejecucion tras instalar numpy, hazlo y corre de nuevo desde aqui.
%pip install -q numpy==2.5.3 pandas==3.0.5 scikit-learn==1.9.1 pyarrow==25.0.1 requests==2.34.2

### 2.3 Código compartido, no duplicado

El entrenamiento (`construir_features`, `construir_pipeline`, `evaluar_gkf`, `comparar_variantes_privacidad`) vive en `entrenar_baseline.py`, en el repo. Es un script normal — su `main()` está protegido por `if __name__ == "__main__":`, así que importarlo no ejecuta nada por sí solo. Este notebook clona el repo a un **tag/commit fijo** (no la rama por defecto, que puede cambiar) e importa esas mismas funciones: nunca copia la lógica.

In [ ]:
# --- Configuracion fija: reemplazar tras publicar el codigo y el release ---
# GITHUB_REPO: "owner/nexplay" del repo publico.
# GITHUB_REF:  tag o commit fijo (NO una rama, NO "latest") — el mismo release que aloja el parquet.
GITHUB_REPO = "REEMPLAZAR_owner/nexplay"
GITHUB_REF = "REEMPLAZAR_tag_fijo"

PARQUET_URL = f"https://github.com/{GITHUB_REPO}/releases/download/{GITHUB_REF}/nexplay_extracto.parquet"
# sha256 real del extracto generado por extracto_datos.py sobre el estado actual de nexplay.db.
# Si se regenera el extracto, este valor tiene que actualizarse junto con el asset del release.
PARQUET_SHA256 = "645d685df884c9a352e4306e899c62768ea493375d4b0378c201c7fc6d2d2d6e"

In [ ]:
!git clone --quiet --branch {GITHUB_REF} --depth 1 https://github.com/{GITHUB_REPO}.git repo_nexplay

In [ ]:
import sys

sys.path.insert(0, "repo_nexplay")

from entrenar_baseline import (  # noqa: E402 (import tras sys.path.insert, a proposito)
    N_SPLITS,
    SEMILLA,
    comparar_variantes_privacidad,
    construir_features,
    construir_pipeline,
    evaluar_gkf,
)

In [ ]:
import hashlib
from pathlib import Path

import requests

DATA_PATH = Path("nexplay_extracto.parquet")

resp = requests.get(PARQUET_URL, timeout=60)
resp.raise_for_status()
DATA_PATH.write_bytes(resp.content)

sha256_obtenido = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if sha256_obtenido != PARQUET_SHA256:
    raise ValueError(
        f"SHA-256 no coincide: esperado {PARQUET_SHA256}, obtenido {sha256_obtenido}. "
        "El asset del release pudo cambiar o la descarga se corrompio; no seguir sin verificarlo."
    )
print(f"descarga verificada: {DATA_PATH.stat().st_size / 1024:.1f} KB, sha256 OK")

In [ ]:
import pandas as pd

df = pd.read_parquet(DATA_PATH)
df["y"] = ((df["playtime_at_review"] < 120) & (df["voted_up"] == 0)).astype(int)

print(f"filas={len(df)}  juegos={df['appid'].nunique()}")
df.head()

## 3. EDA

### 3.1 Prevalencia global

In [ ]:
prevalencia_global = df["y"].mean()
print(f"prevalencia global de arrepentimiento temprano: {prevalencia_global:.4f} ({100*prevalencia_global:.2f}%)")

Clase muy desbalanceada — por eso la métrica de validación es PR-AUC, no accuracy (un modelo que siempre dice "no" acierta más del 97% de las veces sin decir nada útil).

### 3.2 Prevalencia por juego

La Capa A se armó con una hipótesis concreta: juegos duros para un jugador nuevo (Dark Souls, Kenshi, Dwarf Fortress) deberían mostrar más arrepentimiento temprano que puertas de entrada (Stardew Valley, Hades, Portal).

In [ ]:
por_juego = (
    df.groupby("nombre")["y"]
    .agg(n="size", prevalencia="mean")
    .sort_values("prevalencia")
)

duros = ["DARK SOULS™: REMASTERED", "Kenshi", "Dwarf Fortress"]
accesibles = ["Portal", "Stardew Valley", "Hades"]
por_juego.loc[duros + accesibles]

Los tres juegos elegidos por "difíciles para un novato" tienen prevalencia tan baja como los de entrada — entre 0.3% y 1.3%, todos por debajo de la media global. La dificultad del juego para alguien nuevo, sola, no separa nada.

In [ ]:
print("--- menor prevalencia ---")
display(por_juego.head(8))
print("--- mayor prevalencia ---")
display(por_juego.tail(8))

Lo que sí separa con fuerza son lanzamientos con recepción muy pareja/floja frente a la expectativa (*The Lord of the Rings: Gollum*, *WILD HEARTS*, *Redfall*, *Suicide Squad: Kill the Justice League*, *Skull and Bones*, *Battlefield 2042*), con prevalencias entre 10% y 27% — 5 a 12 veces la media global, y muy por encima de cualquier juego "difícil" de la Capa A. Es exactamente la intuición detrás de la Capa C del catálogo (brecha expectativa vs. recepción), no de la Capa A.

**Encontramos que** el eje que separa a los juegos no es qué tan duro es para un jugador nuevo, sino qué tan bien fue recibido en su lanzamiento — algo que se puede leer, en buena medida, del lado del juego (precio, descuento, nota de Metacritic) sin necesitar casi nada del perfil de quien compra. Esto reencuadra el proyecto: en vez de perfilar exhaustivamente al jugador, el modelo debe apoyarse primero en lo que transfiere del juego, y tratar el perfil declarado del jugador como un ajuste secundario — no como el eje principal. Lo confirmamos con números en la sección 6 (Modelo): el conjunto sin ninguna variable de jugador retiene casi todo el PR-AUC del conjunto completo de producción.

## 4. Calidad de datos

In [ ]:
priv_pct = (df["num_games_owned"] == 0).mean()
metacritic_na_pct = df["metacritic"].isna().mean()
precio_na_pct = df["precio_final"].isna().mean()

print(f"perfiles con num_games_owned == 0: {100*priv_pct:.1f}%")
print(f"juegos sin nota de metacritic: {100*metacritic_na_pct:.1f}%")
print(f"filas sin precio_final: {100*precio_na_pct:.1f}%")

**`num_games_owned == 0` es bandera de privacidad, no biblioteca vacía.** Steam no distingue "no tiene juegos" de "su biblioteca es privada" — ambos casos llegan como 0. Con ~60% de las reseñas en ese caso, imputarlo como "cero juegos" sería tratar como información algo que es, en su mayoría, ausencia de información. Por eso el proyecto lo trata con un flag explícito en vez de imputarlo en silencio (ver sección 7 — y por qué ese flag, ya evaluado, no quedó en el modelo de producción).

In [ ]:
# Dos juegos pagos sin precio_final (no son gratis, pero el campo llego nulo en la ingesta):
mask_precio_raro = df["precio_final"].isna() & (df["es_gratis"] == 0)
df.loc[mask_precio_raro, "nombre"].unique()

Son 2 de 83 juegos (3,000 de 123,972 filas, 2.4%) — probablemente removidos o repriceados en Steam entre la ingesta y hoy. `construir_features` ya los trata con `fillna(0)`, igual que a los juegos gratis; no hace falta una regla nueva.

**Nada de fuga temporal.** El extracto no trae `playtime_forever` ni ningún campo que solo exista porque el autor ya reseñó (`num_reviews`, `steam_purchase`, etc.) — esas columnas solo se usan en el conjunto `'completo'` de `entrenar_baseline.py`, que es apenas un chequeo interno de "¿hay señal?" y nunca llega a producción.

In [ ]:
fechas = pd.to_datetime(df["timestamp_created"], unit="s")
print(f"reseñas entre {fechas.min().date()} y {fechas.max().date()}")

assert df["playtime_at_review"].ge(0).all(), "playtime_at_review negativo"
assert df["voted_up"].isin([0, 1]).all(), "voted_up fuera de {0,1}"
assert df["descuento"].dropna().between(0, 100).all(), "descuento fuera de [0,100]"
print("chequeos de rango: OK")

## 5. Ingeniería de variables

`construir_features(df, conjunto="compra")` arma exactamente las seis variables del modelo de producción: solo lo que se conoce **antes** de que el jugador juegue — lo que el catálogo ya sabe del juego, más lo que el formulario de alta declara del jugador. Nada que dependa de que la reseña ya exista.

In [ ]:
X, y, grupos = construir_features(df, conjunto="compra")
print("features:", list(X.columns))
X.assign(y=y).sample(5, random_state=SEMILLA)

- `log_num_games_owned`: `log1p` sobre un conteo con cola larga (unos pocos perfiles declaran cientos de juegos). En producción, `compras_al_anio` del formulario sustituye a `num_games_owned` — es la variable que el jugador sí puede declarar sin depender de una cuenta externa.
- `es_gratis`, `descuento`: ya vienen acotadas (0/1 y 0-100), sin transformar.
- `log_precio_final`: mismo `log1p`, precios van de centavos a cientos de pesos.
- `metacritic_disponible` + `metacritic`: el 27% sin nota se imputa con la mediana, pero marcado con un flag — el modelo puede aprender que "no tiene nota" es distinto de "tiene una nota mediocre", en vez de mezclarlos en silencio.

`grupos` es `appid`: es la clave de todo el esquema de validación de la sección 6.

## 6. Modelo

Regresión logística con `class_weight="balanced"`, sin tuning — el piso que hay que superar, no el modelo final. Validación con `GroupKFold` por `appid`: cada fold deja afuera juegos completos, así el PR-AUC mide generalización a juegos que el modelo nunca vio, no memorización de un juego particular.

In [ ]:
from sklearn.dummy import DummyClassifier

print(f"GroupKFold con N_SPLITS={N_SPLITS}, semilla={SEMILLA}\n")

print("=== conjunto 'compra' (modelo de produccion) ===")
trivial = evaluar_gkf(DummyClassifier(strategy="prior"), X, y, grupos, "compra/trivial")
logreg = evaluar_gkf(construir_pipeline(), X, y, grupos, "compra/logreg")

In [ ]:
print("=== conjunto 'juego' (solo lado del juego, sin nada del jugador) ===")
X_juego, y_juego, grupos_juego = construir_features(df, conjunto="juego")
print("features:", list(X_juego.columns), "\n")

trivial_juego = evaluar_gkf(DummyClassifier(strategy="prior"), X_juego, y_juego, grupos_juego, "juego/trivial")
logreg_juego = evaluar_gkf(construir_pipeline(), X_juego, y_juego, grupos_juego, "juego/logreg")

In [ ]:
caida = 1 - logreg_juego.mean() / logreg.mean()
print(f"compra: PR-AUC={logreg.mean():.4f}  juego: PR-AUC={logreg_juego.mean():.4f}")
print(f"quitar TODO el lado del jugador cuesta solo {100*caida:.1f}% de PR-AUC")

Confirma lo que sugería la sección 3: sacar por completo el lado del jugador (`compras_al_anio` vía `log_num_games_owned`) apenas mueve el PR-AUC. La mayor parte de la señal viene del juego, no de quién compra — coherente con `nota_plataforma` en la API: "el lado del juego transfiere".

In [ ]:
pipeline_final = construir_pipeline()
pipeline_final.fit(X, y)
coefs = pd.Series(pipeline_final.named_steps["clf"].coef_[0], index=X.columns).sort_values()
coefs

`class_weight="balanced"` reescala las clases para que el modelo aprenda con la minoría, pero eso significa que `predict_proba` **ya no es una probabilidad calibrada** — es un score útil para *ordenar* riesgo relativo, no para leerse como "38% de probabilidad de arrepentimiento". Por eso la API (`api/scoring.py`) y la UI solo exponen un nivel (BAJO/MEDIO/ALTO, calibrado por tercios de la distribución de scores de validación) y nunca un porcentaje — mostrarlo como probabilidad induciría a error.

## 7. Experimento de privacidad

`privacidad_perfil` (`num_games_owned == 0`, sección 4) se evaluó como feature explícita del conjunto `'compra'` antes de que existiera el modelo de producción. `comparar_variantes_privacidad` reproduce esa comparación ya aprobada, sin volver a experimentar: tres variantes con `GroupKFold` sobre las mismas features de `'compra'` más/menos esa bandera.

In [ ]:
resultados_privacidad = comparar_variantes_privacidad(df)

In [ ]:
for variante in ("con_privacidad", "sin_privacidad", "solo_publico"):
    r = resultados_privacidad[variante]
    print(f"{variante:<15} media={r.mean():.4f}  std={r.std():.4f}")

diferencia = resultados_privacidad["con_privacidad"].mean() - resultados_privacidad["sin_privacidad"].mean()
desviacion = resultados_privacidad["con_privacidad"].std()
print(f"\ndiferencia con/sin bandera: {diferencia:.4f}")
print(f"desviacion entre folds: {desviacion:.4f}")
print(f"tamaño de 'solo_publico' (privacidad_perfil == 0): {resultados_privacidad['n_solo_publico']} filas")

La diferencia de PR-AUC entre incluir la bandera y no incluirla (~0.0036) es un orden de magnitud menor que la desviación entre folds (~0.027): no hay señal real, es ruido de muestreo. El subconjunto `solo_publico` (perfiles no privados, 40% de las filas) además muestra folds muy inestables (0.014 a 0.156) por tener menos positivos por fold — otra razón para no construir una regla especial alrededor de él.

**Decisión ya tomada y aplicada:** `privacidad_perfil` se sacó del conjunto `'compra'` en `entrenar_baseline.py` (se conserva solo en `'completo'`, donde se originó la comparación) y `modelo/nexplay.pkl` se re-entrenó sin ella — es el modelo que corre en `api/scoring.py` hoy.

## 8. Conclusiones

- **El target es una señal proxy, no arrepentimiento observado.** `playtime_at_review < 120` y `voted_up == 0` es lo más cercano que da la API de Steam a "esto no era lo que esperaba", pero no es lo mismo que preguntarle al jugador.
- **El reencuadre central de este proyecto:** el riesgo depende mucho más del juego (precio, descuento, recepción de crítica) que de quién lo compra. El conjunto `'juego'` —sin ninguna variable de jugador— retiene casi todo el PR-AUC del conjunto de producción `'compra'`. El perfil declarado en el formulario de alta aporta, pero es un ajuste secundario, no el eje principal del riesgo.
- **`privacidad_perfil` se descartó con evidencia, no por intuición**: la diferencia de PR-AUC al quitarla es un orden de magnitud menor que el ruido entre folds.
- **El modelo de producción (`modelo/nexplay.pkl`) es un piso, no un techo**: regresión logística sin tuning, seis variables, PR-AUC ~0.07 contra una prevalencia de 2.2% (~3-4x mejor que un clasificador trivial). Con `class_weight="balanced"` el score ordena riesgo relativo pero no es una probabilidad calibrada — por eso la API expone un nivel (BAJO/MEDIO/ALTO, por tercios de la distribución de validación) y no un porcentaje.
- **Límites conocidos:** todo el entrenamiento es de reseñas de Steam (PC); no existe una fuente propia de PlayStation/Xbox/Nintendo, así que el lado del juego transfiere a otras plataformas pero el modelo no fue validado ahí (`nota_plataforma` en la API lo advierte). Tampoco se usó texto de reseña ni el corpus de Metacritic (fuente secundaria, solo para comparar motivos, nunca para entrenar).
- **Próximo paso natural:** el conjunto `'completo'` (con `num_reviews`, etc.) muestra que hay algo más de señal cuando se conocen datos posteriores a la reseña — pero eso es fuga en producción. Vale la pena explorar features de texto o de comportamiento temprano dentro de la ventana de reembolso, no post-hoc.